# Exploratory Data Analysis (EDA)

## Table of Contents
1. [Dataset Overview](#dataset-overview)
2. [Handling Missing Values](#handling-missing-values)
3. [Feature Distributions](#feature-distributions)
4. [Possible Biases](#possible-biases)
5. [Correlations](#correlations)


. [Correlations](#correlations)


In [3]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


## Dataset Overview

[Provide a high-level overview of the dataset. This should include the source of the dataset, the number of samples, the number of features, and example showing the structure of the dataset.]


Data from https://tatoeba.org via https://www.manythings.org/anki/
Tatoeba.org is a large database of example sentences translated into many languages by its members who volunteer their time.
Since this a work of volunteers in their free time there are mistakes:
- There are sentences that are grammatically wrong.
- There are sentences that don't sound natural even though they are grammatically correct
- There are incorrect translations.

The authors of manythings.org/anki combined the english sentences with other, e.x. Portuguese, languages into txt files. To reduce the amount of errors the authors of manythings.org/anki did a clean up: "In order to minimize the number of errors, I only used sentences that were owned by identified native speakers working on the Tatoeba Project and English sentences that I've personally checked and did not reject." While this reduces errors it does reduce them to zero.

In [4]:
import pandas as pd

def sep():
    print("-" * 40)

# Load the data
# Replace 'your_dataset.csv' with the path to your actual dataset
df = pd.read_csv("../por.txt", sep="\t", header=None, names=["english", "portuguese", "attribution"])

# Number of samples
num_samples = df.shape[0]


# Display these dataset characteristics
print(f"Number of samples: {num_samples}")
sep()
# Number of features not really applicable 


# Display the first few rows of the dataframe to show the structure
print("Example data:")
print(df.head())
sep()

# tatoeba requires attribution to all the individual contributors
contributors = df["attribution"].str.findall(r"#\d+ \((\w+)\)").explode().unique().tolist()

print("Example of individual contributors:")
print(contributors[0:5])
sep()

# Drop attribution for the rest of exploration:
df = df[["english", "portuguese"]]
print("Example data without attribution:")
print(df.head())
sep()
# The sentences are ordered by length so also show random sample
print("Random samples:")
print(df.sample(5))

Number of samples: 197147
----------------------------------------
Example data:
  english portuguese                                        attribution
0     Go.       Vai.  CC-BY 2.0 (France) Attribution: tatoeba.org #2...
1     Go.        Vá.  CC-BY 2.0 (France) Attribution: tatoeba.org #2...
2     Hi.        Oi.  CC-BY 2.0 (France) Attribution: tatoeba.org #5...
3    Run!     Corre!  CC-BY 2.0 (France) Attribution: tatoeba.org #9...
4    Run!     Corra!  CC-BY 2.0 (France) Attribution: tatoeba.org #9...
----------------------------------------
Example of individual contributors:
['CM', 'alexmarcelo', 'ToinhoAlam', 'brauliobezerra', 'papabear']
----------------------------------------
Example data without attribution:
  english portuguese
0     Go.       Vai.
1     Go.        Vá.
2     Hi.        Oi.
3    Run!     Corre!
4    Run!     Corra!
----------------------------------------
Random samples:
                                            english  \
139873           Tom is a victi

## Handling Missing Values

[Identify any missing values in the dataset, and describe your approach to handle them if there are any. If there are no missing values simply indicate that there are none.]


In [5]:
# Check for missing values
missing_values = df.isnull().sum()
missing_values


english       0
portuguese    0
dtype: int64

To reduce complexity all words will be lowercase and non ascii characters will be normalized to ascii. ? and ! carry a special meaning and will be treated as a word, all other non letters are removed.

In [6]:
import unicodedata
import re

def unicode_to_ascii(input_str):
    nfkd_form = unicodedata.normalize('NFKD', input_str)
    return u"".join([c for c in nfkd_form if not unicodedata.combining(c)])
    
def normalize_string(s):
    s = unicode_to_ascii(s.lower().strip())
    s = re.sub(r"([!?.])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()

df = df.map(normalize_string)
df.sample(5)

,english,portuguese
158141,it s because of you that we were late,chegamos atrasados por sua causa
195377,maybe eventually you ll decide you don t want ...,talvez no final voce decida que nao quer mais ...
76805,what color is your dress ?,qual e a cor do teu vestido ?
150918,tom should spend more time studying,tom deveria passar mais tempo estudando
176278,i m going to do that the way tom told me to,vou fazer isso do jeito que tom me disse


To analyze the data further, build up word counts for Portuguese and English.

In [22]:
from collections import Counter

eng = Counter()
por = Counter()

for _, row in df.iterrows():
  eng.update(row["english"].split())
  por.update(row["portuguese"].split())

In [24]:
print(len(eng), len(por))


13101 22150


Big difference, possible explaination
Portuguese is a heavily inflected language — the same verb has many different surface forms:
- English: I go, you go, he goes → 2 forms (go, goes)
- Portuguese: eu vou, tu vais, ele vai, nos vamos, vos ides, eles vao → 6 forms
words differ based on gender:
thank you -> obrigado/obrigada based on the gender of the speaking person

In [23]:
print(sorted(eng.items(), key=lambda x: x[1], reverse=True)[:10])
print(sorted(por.items(), key=lambda x: x[1], reverse=True)[:10])


[('i', 65737), ('tom', 54639), ('you', 45782), ('to', 45104), ('the', 35290), ('?', 32865), ('t', 31913), ('a', 24538), ('that', 22722), ('is', 22033)]
[('tom', 54426), ('que', 45871), ('o', 41848), ('nao', 39532), ('eu', 36432), ('?', 33019), ('de', 30712), ('a', 29913), ('e', 27471), ('voce', 26892)]


Notable: count for tom is not exactly the same: sometimes the portuguese sentence uses tomas or antonio instead of tom:

In [13]:
df[df["english"].str.count(r"\btom\b") != df["portuguese"].str.count(r"\btom\b")].sample(5)

,english,portuguese
113596,tom is older than your father,o tomas e mais velho que o teu pai
188905,tom s father who is in prison never writes to tom,o pai do tom que esta na cadeia nunca lhe escreve
127998,tom tells us jokes all the time,antonio nos conta piadas o tempo todo
155093,tom invited us to his summer cottage,o tomas convidou nos a visitar a casa de verao...
189628,tom didn t realize that mary didn t know how t...,antonio nao percebeu que maria nao sabia como ...


High count for "t" in english. Normalization splits "don't" into "don" and "t". Model will probably learn that don and t together mean negation. If not maybe expand
    "don't": "do not",
      "can't": "cannot",
      "won't": "will not",
      "i'm": "i am",
      "i've": "i have",
      "it's": "it is"
but that would need to be exhaustive. For now leave it as is.

## Feature Distributions

[Plot the distribution of various features and target variables. Comment on the skewness, outliers, or any other observations.]


In [2]:
# not applicable

## Possible Biases

[Investigate the dataset for any biases that could affect the model’s performance and fairness (e.g., class imbalance, historical biases).]


The dataset was originally curated for human learners, so the language tends to be simple and more complex words and phrases are sparse.

## Correlations

[Explore correlations between features and the target variable, as well as among features themselves.]


In [1]:
# not applicable